# PHẦN 3: KỸ THUẬT ĐẶC TRƯNG VÀ QUẢN LÝ BỘ NHỚ


In [1]:
import os, pandas as pd, numpy as np, gc, pyarrow as pa, pyarrow.parquet as pq, sqlite3

In [2]:
IS_SAMPLE = False
NEGATIVE_RATIO = 4
CHUNK_SIZE = 500000
PROCESSED_DATA_DIR = '../data/processed'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.csv')
META_PATH = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.csv')
CAND_PATH = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.csv')
FEAT_OUT = os.path.join(PROCESSED_DATA_DIR, 'features.parquet')

In [3]:
dtypes_cand = {
    'mapped_user_id': 'int32', 
    'mapped_item_id': 'int32', 
    'sasrec_rank': 'float32', 
    'lightgcn_rank': 'float32'
}
df_cands_ranks = pd.read_csv(CAND_PATH, dtype=dtypes_cand)

df_train = pd.read_csv(TRAIN_PATH)
df_meta = pd.read_csv(META_PATH, low_memory=False)

def downcast(df):
    for c in df.select_dtypes(include=['float64']).columns: df[c] = df[c].astype('float32')
    for c in df.select_dtypes(include=['int64']).columns: df[c] = df[c].astype('int32')
    return df

mapping_df = df_train[['parent_asin', 'mapped_item_id']].drop_duplicates()
df_meta = pd.merge(df_meta, mapping_df, on='parent_asin', how='inner')
df_meta['price'] = pd.to_numeric(df_meta['price'], errors='coerce').fillna(0)

df_meta = downcast(df_meta)
df_train = downcast(df_train)

user_stats = df_train.groupby('mapped_user_id').size().reset_index(name='user_orders')
item_stats = df_train.groupby('mapped_item_id').size().reset_index(name='item_sales')
popular_items = item_stats.sort_values('item_sales', ascending=False)['mapped_item_id'].values

if os.path.exists(FEAT_OUT): os.remove(FEAT_OUT)

users = df_train['mapped_user_id'].unique()
conn = sqlite3.connect('train_db.sqlite')
df_train[['mapped_user_id', 'mapped_item_id']].to_sql('interactions', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX idx_user ON interactions(mapped_user_id)')

writer = None
for start in range(0, len(users), CHUNK_SIZE):
    chunk_u = users[start:start+CHUNK_SIZE]
    df_chunk_pos = df_train[df_train['mapped_user_id'].isin(chunk_u)][['mapped_user_id', 'mapped_item_id']].copy()
    df_chunk_pos['label'] = 1
    
    neg_records = []
    
    users_tup = tuple(chunk_u)
    cur = conn.execute(f"SELECT mapped_user_id, mapped_item_id FROM interactions WHERE mapped_user_id IN ({','.join(['?']*len(chunk_u))})", users_tup)
    rows = cur.fetchall()
    interacted_dict = {}
    for r in rows:
        interacted_dict.setdefault(r[0], set()).add(r[1])
        
    for u in chunk_u:
        interacted = interacted_dict.get(u, set())
        neg_needed = len(interacted) * NEGATIVE_RATIO
        negs = []
        for p in popular_items:
            if p not in interacted: negs.append(p)
            if len(negs) >= neg_needed: break
        for n in negs: neg_records.append({'mapped_user_id': u, 'mapped_item_id': n, 'label': 0})
            
    df_chunk = pd.concat([df_chunk_pos, pd.DataFrame(neg_records)], ignore_index=True)
    df_chunk = pd.merge(df_chunk, user_stats, on='mapped_user_id', how='left')
    df_chunk = pd.merge(df_chunk, item_stats, on='mapped_item_id', how='left')
    df_chunk = pd.merge(df_chunk, df_meta[['mapped_item_id', 'price']].drop_duplicates(), on='mapped_item_id', how='left')
    df_chunk = pd.merge(df_chunk, df_cands_ranks, on=['mapped_user_id', 'mapped_item_id'], how='left')
    
    # KHÔNG FULL NA CHO RANK (sasrec_rank, lightgcn_rank) ĐỂ XGBOOST XỬ LÝ MISSING.
    df_chunk.fillna({'price': 0, 'user_orders': 0, 'item_sales': 0}, inplace=True)
    df_chunk = downcast(df_chunk)
    
    table = pa.Table.from_pandas(df_chunk)
    if writer is None: writer = pq.ParquetWriter(FEAT_OUT, table.schema)
    writer.write_table(table)
    
    del df_chunk, df_chunk_pos, neg_records, table
    gc.collect()

if writer is not None: writer.close()
conn.close()
os.remove('train_db.sqlite')
print("Feature Engineering chunk loop completed.")



Feature Engineering chunk loop completed.
